# 长文先整体编码再分块

普通做法先切出片段，再让向量模型分别读取。片段太短时，模型可能看不到解释它的前后文。Late Chunking 调换了顺序：先让模型读取一段连续文字，再从模型输出中取出目标片段对应的 token，合成这个片段的向量。

下面使用教程已有的 `BAAI/bge-small-zh-v1.5` 做一组短上下文对照：固定选择 1 道主要问题和 1 道复查问题，分别从原始 PDF 正常检索前 10 条、每条约 320 字的片段；只有候选和排序完成后才读取标注核对必要页。这样不会用答案标注挑选候选。模型最多处理 512 个 token，因此这里只验证方法的基本动作，不把它写成已经完成了整本书或超长文档编码。Notebook 已保存输出；重跑时只读取预先准备的本地模型缓存，缓存缺失会直接报错。

In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', message='IProgress not found.*')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
from transformers import AutoModel, AutoTokenizer

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
    raise FileNotFoundError('没有找到教程数据目录，请从本节所在目录运行。')

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import (
    build_bm25_chunk_search, find_local_bge_model, load_query_catalog, load_pdf_pages, make_fixed_chunks,
)
from common.nontraining_utils import load_annotation

cases = {item['id']: item for item in load_query_catalog()}
page_rows = load_pdf_pages()
pages = {item['page']: item['text'] for item in page_rows}
raw_chunks = make_fixed_chunks(page_rows, chunk_size=320, overlap=0)
chunk_search = build_bm25_chunk_search(raw_chunks)

model_source = find_local_bge_model()
tokenizer = AutoTokenizer.from_pretrained(model_source, local_files_only=True)
model = AutoModel.from_pretrained(model_source, local_files_only=True).eval()

def normalize(vector):
    return torch.nn.functional.normalize(vector, dim=0)

def encode_separately(text):
    inputs = tokenizer(text, truncation=True, max_length=512, return_tensors='pt')
    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state[0]
    return normalize(hidden[0])

def encode_with_surrounding_text(page, chunk):
    page_text = pages[page]
    match_start = page_text.index(chunk)
    context_start = max(0, match_start - 220)
    context_end = min(len(page_text), match_start + len(chunk) + 220)
    parent = page_text[context_start:context_end]
    chunk_start = match_start - context_start
    chunk_end = chunk_start + len(chunk)
    inputs = tokenizer(parent, return_offsets_mapping=True, truncation=True, max_length=512, return_tensors='pt')
    offsets = inputs.pop('offset_mapping')[0]
    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state[0]
    belongs_to_chunk = (offsets[:, 1] > chunk_start) & (offsets[:, 0] < chunk_end) & (offsets[:, 1] > offsets[:, 0])
    if not belongs_to_chunk.any():
        raise ValueError('片段超出模型可读取的范围。')
    return normalize(hidden[belongs_to_chunk].mean(dim=0))

def candidates_from_retrieval(question, top_k=10):
    # 候选只来自原始 PDF 的正常检索；问题集标注在比较完成后才读取。
    hits = chunk_search(question, top_k=top_k)
    return [{'page': hit.pages[0], 'chunk': hit.text} for hit in hits]

def compare(question, candidates):
    query_vector = encode_separately(question)
    rows = []
    for item in candidates:
        separate = encode_separately(item['chunk'])
        with_context = encode_with_surrounding_text(item['page'], item['chunk'])
        rows.append({**item, 'separate_score': float(query_vector @ separate), 'context_score': float(query_vector @ with_context)})
    before = sorted(rows, key=lambda row: row['separate_score'], reverse=True)
    after = sorted(rows, key=lambda row: row['context_score'], reverse=True)
    return before, after

def show(ranking, score_key):
    return [(row['page'], round(row[score_key], 4)) for row in ranking]

def rank_of(page, ranking):
    return next((index for index, row in enumerate(ranking, 1) if row['page'] == page), None)

main = cases['late_chunking_model_evaluation_methods']
main_candidates = candidates_from_retrieval(main['query'], top_k=10)
before, after = compare(main['query'], main_candidates)
main_annotation = load_annotation(main['id'])
target_page = main_annotation['expected_pages'][0]
print('主要问题：', main['query'])
print('片段单独编码（页码，分数）：', show(before, 'separate_score'))
print('连同真实前后文编码（页码，分数）：', show(after, 'context_score'))
print('必要页排名：', rank_of(target_page, before), '→', rank_of(target_page, after))
assert len(main_candidates) == 10 and len(before) == len(after) == 10

check = cases['agentic_random_forest']
check_candidates = candidates_from_retrieval(check['query'], top_k=10)
check_before, check_after = compare(check['query'], check_candidates)
check_annotation = load_annotation(check['id'])
check_page = check_annotation['expected_pages'][0]
print('\n复查问题：', check['query'])
print('片段单独编码（页码，分数）：', show(check_before, 'separate_score'))
print('连同真实前后文编码（页码，分数）：', show(check_after, 'context_score'))
print('必要页排名：', rank_of(check_page, check_before), '→', rank_of(check_page, check_after))
assert len(check_candidates) == 10 and len(check_before) == len(check_after) == 10


主要问题： 第2.2节的三种评估方法是什么？
片段单独编码（页码，分数）： [(181, 0.5947), (3, 0.568), (18, 0.5496), (45, 0.5357), (110, 0.4946), (18, 0.4883), (18, 0.485), (119, 0.4738), (139, 0.4669), (139, 0.4328)]
连同真实前后文编码（页码，分数）： [(18, 0.4627), (45, 0.4526), (181, 0.4424), (18, 0.4373), (110, 0.4341), (139, 0.4051), (18, 0.4031), (139, 0.4023), (119, 0.3885), (3, 0.2674)]
必要页排名： 3 → 1



复查问题： 随机森林的两个核心要点是什么？
片段单独编码（页码，分数）： [(100, 0.5625), (7, 0.5587), (7, 0.5007), (104, 0.4898), (181, 0.4432), (104, 0.4148), (26, 0.4085), (48, 0.3905), (149, 0.3894), (129, 0.3822)]
连同真实前后文编码（页码，分数）： [(100, 0.471), (104, 0.3524), (181, 0.3457), (104, 0.3336), (26, 0.3121), (129, 0.3119), (149, 0.311), (48, 0.3085), (7, 0.162), (7, 0.156)]
必要页排名： 1 → 1


本页只运行两道题：评估方法问题作为主要案例，随机森林问题作为复查案例。两题都先由同一个 BM25 检索器取前 10 条、每条约 320 字，再在各自同一批候选上比较两种片段编码方式。运行时会直接打印必要页在候选中的覆盖和排名；必要页没有被检索到时就显示“未出现”，不借助答案标注补入候选。

这组结果只说明在当前检索到的十个候选片段上，前后文编码是否改变排序；它不是整本书的效果结论。普通 BGE 片段向量取开头 token；Late Chunking 从连续前后文的整体输出中汇总片段 token，因此对照包含了输入范围和片段向量取法两项变化。真正处理超过 512 token 的连续长文，还需要换成长上下文向量模型，并重新检查速度、显存和检索效果。

In [2]:
# 代码要点：model_hidden_states 是长文编码器返回的 [tokens, hidden] 矩阵；
# spans 是分块对应的 token 区间。此格不下载模型，也不写入当前输出。
def late_chunking_pool(model_hidden_states, spans):
    """对连续长文编码后，再按 token span 平均池化每个片段。"""
    vectors = []
    for start, end in spans:
        if end <= start: raise ValueError("chunk span 必须满足 end > start")
        vectors.append(model_hidden_states[start:end].mean(axis=0))
    return vectors

# 生产实现还要做 attention mask、最大 token 长度、跨文档边界和归一化；
# 它不是当前已保存 BGE 短上下文实验的替代品。


## 代码要点与本次结果

`late_chunking_pool(model_hidden_states, spans)` 单独展示按 token span 聚合的核心动作；生产实现还要处理 attention mask、跨文档边界、最大 token 数、归一化、显存和批量大小。当前实验加载器显式要求本地 `BAAI/bge-small-zh-v1.5` 缓存并设置 `local_files_only=True`，不会在运行闭环中隐式联网。

Late Chunking（延迟分块）先编码连续文本，再按 chunk 的 token 边界池化；与检索后补文字的 Sentence Window、父子片段不同，它在索引阶段改变片段向量的语境。在第 2.2 节评估方法问题上，必要页从第 3 升到第 1；随机森林复查题的必要页仍在前十。这里报告的是同一批候选上的实际排序变化，实验边界见上一节。

In [3]:
from common.eval_utils import emit_tutorial_audit

# 统一保存契约：先完成两种编码，再读取 expected_pages 计算排名。
import json

def _actual_pages(items):
    pages = []
    for item in items:
        page = int(item['page'] if isinstance(item, dict) else item.page)
        if page not in pages:
            pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank,
            'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _emit(role, case_id, before_items, after_items, purpose=None):
    annotation = load_annotation(case_id)
    payload = {'case_id': case_id, 'method': '先看上下文再生成片段向量（Late Chunking）', 'role': role,
              'before': _metrics(before_items, annotation['expected_pages']),
              'after': _metrics(after_items, annotation['expected_pages'])}
    if purpose:
        payload['check_purpose'] = purpose
    emit_tutorial_audit(payload)

_emit('main', 'late_chunking_model_evaluation_methods', before, after)
_emit('check', 'agentic_random_forest', check_before, check_after, '确认没有改坏')


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[进入评估](../7.%20评估/端到端验收.ipynb)

